# Customer Churn EDA — IBM Telco Dataset

Exploratory Data Analysis notebook for the Customer Churn Prediction Dashboard.
This notebook is **read-only analysis** — it does not modify the raw data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

df_raw = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Data types and missing values
print('Data Types:')
print(df_raw.dtypes)
print('\nMissing Values:')
print(df_raw.isnull().sum())
print('\nTotalCharges whitespace rows:', (df_raw['TotalCharges'].str.strip() == '').sum())

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df_raw['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#22c55e', '#ef4444'], startangle=90)
axes[0].set_title('Churn Distribution')

sns.countplot(data=df_raw, x='Churn', ax=axes[1], palette={'No': '#22c55e', 'Yes': '#ef4444'})
axes[1].set_title('Churn Count')
plt.tight_layout()
plt.show()
print(churn_counts.to_dict())

In [ ]:
# Churn by key categorical features
cat_cols = ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.flatten(), cat_cols):
    churn_pct = df_raw.groupby(col)['Churn'].apply(lambda x: (x=='Yes').mean() * 100)
    churn_pct.sort_values().plot(kind='barh', ax=ax, color='#3b82f6')
    ax.set_title(f'Churn Rate by {col} (%)')
    ax.set_xlabel('Churn Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Numerical distributions by churn
df_clean = df_raw.copy()
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
for ax, col in zip(axes, num_cols):
    for label, color in [('No', '#22c55e'), ('Yes', '#ef4444')]:
        subset = df_clean[df_clean['Churn'] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color)
    ax.set_title(f'{col} by Churn')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap on numeric features
numeric_df = df_clean[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
numeric_df['Churn_int'] = (df_clean['Churn'] == 'Yes').astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Feature engineering preview
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from model.preprocessing import clean_data, engineer_features, prepare_dataset

df_feat, X, y, feature_cols, val_report = prepare_dataset('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print('Validation Report:', val_report)
print('\nEngineered features preview:')
df_feat[['customerID', 'tenure', 'AvgMonthlyCharge', 'TotalServices', 
          'HasMultipleServices', 'TenureGroup', 'ContractRisk', 'Churn']].head(10)

In [ ]:
# Model evaluation (requires trained model)
import json
try:
    with open('../model/model_metrics.json') as f:
        metrics = json.load(f)
    print('=== Logistic Regression ===')
    lr = metrics['logistic_regression']
    print(f"  Accuracy : {lr['accuracy']:.4f}")
    print(f"  Precision: {lr['precision']:.4f}")
    print(f"  Recall   : {lr['recall']:.4f}")
    print(f"  F1       : {lr['f1']:.4f}")
    print(f"  ROC-AUC  : {lr['roc_auc']:.4f}")
    print('\n=== XGBoost ===')
    xgb = metrics['xgboost']
    print(f"  Accuracy : {xgb['accuracy']:.4f}")
    print(f"  Precision: {xgb['precision']:.4f}")
    print(f"  Recall   : {xgb['recall']:.4f}")
    print(f"  F1       : {xgb['f1']:.4f}")
    print(f"  ROC-AUC  : {xgb['roc_auc']:.4f}")
    print(f"\nBest model: {metrics['best_model']}")
except FileNotFoundError:
    print('Run python model/train_model.py first to generate metrics.')